In [2]:
# ============================================================
# 02_preprocessing_and_split.ipynb — Cell 1: Setup & Configuration
# ============================================================
# Part B of the Digikala project: cleaning, splitting, masking,
# and production of standard artifacts.
#
# This cell does NOT read, clean, or write any data.
# It only:
#   - imports libraries
#   - defines shared paths and constants
#   - verifies that input files and required packages are present
# ============================================================

from __future__ import annotations

# --- Standard library --------------------------------------------------------
import hashlib   # for the deterministic masking protocol (B4)
import json      # for split_manifest.json (B5)
import random    # for any bounded sampling, always with a seed
from pathlib import Path

# --- Third-party: data processing --------------------------------------------
import polars as pl

# --- Third-party: Persian NLP ------------------------------------------------
import hazm      # normalizer for Persian text (B1)

# --- Reproducibility seeds (fixed by the Definition, must not change) ------
SPLIT_SEED = 2026
MASK_SEED = "digikala-semi-supervised-v1"
MASK_RATE = 0.20

# --- Project layout ----------------------------------------------------------
PROJECT_ROOT = Path("/home/jvd/Desktop/persian-nlp-pipeline")
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR    = PROJECT_ROOT / "data" / "splits"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

COMMENTS_CSV = RAW_DIR / "digikala-comments.csv"
PRODUCTS_CSV = RAW_DIR / "digikala-products.csv"

# --- Output files that Part B will produce -----------------------------------
# (declared here as a single source of truth, populated in later cells)
COMMENTS_CLEAN_PARQUET = PROCESSED_DIR / "comments_clean.parquet"
TRAIN_SUPERVISED_CSV   = SPLITS_DIR / "train_supervised.csv"
TRAIN_PSEUDO_UNLABELED_CSV = SPLITS_DIR / "train_pseudo_unlabeled.csv"
VALIDATION_CSV         = SPLITS_DIR / "validation.csv"
TEST_CSV               = SPLITS_DIR / "test.csv"
SPLIT_MANIFEST_JSON    = SPLITS_DIR / "split_manifest.json"

# --- Create output directories if they do not exist --------------------------
for _d in (PROCESSED_DIR, SPLITS_DIR, ARTIFACTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Fail-fast: input files must exist --------------------------------------
for _p in (COMMENTS_CSV, PRODUCTS_CSV):
    assert _p.exists(), f"Missing input file: {_p}"

# --- Fail-fast: hazm must import (installed in the uv environment) ----------
# The version string is handy when the normalizer's behavior changes across
# hazm releases.
_hazm_version = getattr(hazm, "__version__", "unknown")

# --- Ready message -----------------------------------------------------------
print("Part B environment ready.")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"COMMENTS_CSV   : {COMMENTS_CSV}  "
      f"({COMMENTS_CSV.stat().st_size / 1024**2:,.1f} MB)")
print(f"PRODUCTS_CSV   : {PRODUCTS_CSV}  "
      f"({PRODUCTS_CSV.stat().st_size / 1024**2:,.1f} MB)")
print(f"PROCESSED_DIR  : {PROCESSED_DIR}")
print(f"SPLITS_DIR     : {SPLITS_DIR}")
print(f"ARTIFACTS_DIR  : {ARTIFACTS_DIR}")
print()
print(f"SPLIT_SEED     : {SPLIT_SEED}")
print(f"MASK_SEED      : {MASK_SEED!r}")
print(f"MASK_RATE      : {MASK_RATE}")
print()
print(f"polars version : {pl.__version__}")
print(f"hazm version   : {_hazm_version}")
print()
print("Planned outputs (will be written by later cells):")
for p in (
    COMMENTS_CLEAN_PARQUET,
    TRAIN_SUPERVISED_CSV,
    TRAIN_PSEUDO_UNLABELED_CSV,
    VALIDATION_CSV,
    TEST_CSV,
    SPLIT_MANIFEST_JSON,
):
    print(f"  {p.relative_to(PROJECT_ROOT)}")

Part B environment ready.
PROJECT_ROOT   : /home/jvd/Desktop/persian-nlp-pipeline
COMMENTS_CSV   : /home/jvd/Desktop/persian-nlp-pipeline/data/raw/digikala-comments.csv  (1,219.3 MB)
PRODUCTS_CSV   : /home/jvd/Desktop/persian-nlp-pipeline/data/raw/digikala-products.csv  (246.7 MB)
PROCESSED_DIR  : /home/jvd/Desktop/persian-nlp-pipeline/data/processed
SPLITS_DIR     : /home/jvd/Desktop/persian-nlp-pipeline/data/splits
ARTIFACTS_DIR  : /home/jvd/Desktop/persian-nlp-pipeline/artifacts

SPLIT_SEED     : 2026
MASK_SEED      : 'digikala-semi-supervised-v1'
MASK_RATE      : 0.2

polars version : 2.1.0
hazm version   : unknown

Planned outputs (will be written by later cells):
  data/processed/comments_clean.parquet
  data/splits/train_supervised.csv
  data/splits/train_pseudo_unlabeled.csv
  data/splits/validation.csv
  data/splits/test.csv
  data/splits/split_manifest.json
